# Rice Pest Detection with YOLOv8
**Deteksi Hama Padi Berbasis YOLOv8 — Desi Susanti, S.Kom., M.Kom.**

Notebook ini melatih model deteksi objek **YOLOv8** untuk mengenali **14 jenis hama padi** dari dataset publik **IP102** (Wu dkk., CVPR 2019), lalu membandingkan dua ukuran model (YOLOv8n dan YOLOv8s), dan membuat demo web dengan **Gradio**.

| Langkah | Isi |
|---|---|
| 0 | Menyiapkan GPU |
| 1 | Memasang pustaka |
| 2 | Menyiapkan dataset IP102 |
| 3 | Mengubah anotasi VOC → YOLO (hanya 14 kelas hama padi) |
| 4 | Eksplorasi data |
| 5 | Melatih YOLOv8n dan YOLOv8s |
| 6 | Evaluasi dan perbandingan |
| 7 | Contoh hasil deteksi |
| 8 | Menyimpan hasil untuk GitHub |
| 9 | Demo Gradio |

> **Catatan lisensi data:** IP102 bebas dipakai untuk keperluan akademik. Dataset **tidak** ikut diunggah ke GitHub; yang diunggah hanya kode, hasil, dan bobot model.

## Langkah 0 — Menyiapkan GPU
Di Colab: menu **Runtime › Change runtime type › T4 GPU**, lalu **Save**. Setelah itu jalankan sel di bawah.

Notebook ini juga bisa dijalankan di **Jupyter/VS Code di Mac**: model otomatis memakai `mps` (GPU Apple), tetapi pelatihan jauh lebih lambat daripada Colab.

In [ ]:
import os, sys, platform, subprocess
IN_COLAB = "google.colab" in sys.modules
try:
    import torch
    DEVICE = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
    print("PyTorch", torch.__version__, "| perangkat:", DEVICE)
    if DEVICE == 0:
        print("GPU:", torch.cuda.get_device_name(0))
except ImportError:
    DEVICE = "cpu"
    print("PyTorch belum terpasang; jalankan Langkah 1 dulu.")
print("Colab:", IN_COLAB, "| Python", platform.python_version())

## Langkah 1 — Memasang pustaka
- **ultralytics**: pustaka resmi YOLOv8 (lisensi AGPL-3.0).
- **gradio**: membuat antarmuka web demo.

In [ ]:
%pip install -q "ultralytics>=8.3,<8.4" "gradio>=4.44" "gdown>=5.2"
import torch
DEVICE = 0 if torch.cuda.is_available() else ("mps" if torch.backends.mps.is_available() else "cpu")
import ultralytics
ultralytics.checks()

## Langkah 2 — Menyiapkan dataset IP102
Pilih salah satu cara dengan mengisi `USE_DRIVE`:

- **`USE_DRIVE = False` (bawaan, tanpa Google Drive):** dataset diunduh langsung dari folder resmi IP102
  (https://github.com/xpwu95/IP102) ke Colab. Hasil pelatihan disimpan sementara di Colab, lalu diunduh sebagai **zip** di akhir.
  Kalau Colab terputus, hasil pelatihan ikut hilang, jadi biarkan tab tetap terbuka.
- **`USE_DRIVE = True`:** dataset dibaca dari pintasan folder *Detection* di Google Drive Anda, dan hasil pelatihan disimpan di Drive.
  Pakai cara ini hanya kalau izin Google Drive berhasil.

In [ ]:
from pathlib import Path
USE_DRIVE = False
ARCHIVE_EXT = (".tar", ".tar.gz", ".tgz", ".zip")
IP102_FOLDER = "https://drive.google.com/drive/folders/1svFSy2Da3cVMvekBwe13mzyx38XZ9xWo"

if IN_COLAB:
    LOCAL = Path("/content")
    if USE_DRIVE:
        from google.colab import drive
        drive.mount("/content/drive")
        DRIVE = Path("/content/drive/MyDrive")
    else:
        DRIVE = LOCAL
else:
    DRIVE = Path.home() / "rice-pest-drive"      # ganti sesuai lokasi di Mac
    LOCAL = Path.home() / "rice-pest-local"

WORK = DRIVE / "rice-pest-yolov8"
WORK.mkdir(parents=True, exist_ok=True)
DATA_SOURCE = None    # isi manual bila perlu, misalnya Path("/content/Detection.tar")

def find_detection_source(root, max_depth=4):
    root = Path(root)
    for dirpath, dirnames, filenames in os.walk(root):
        depth = len(Path(dirpath).relative_to(root).parts)
        if depth > max_depth:
            dirnames[:] = []
            continue
        for name in dirnames + filenames:
            low = name.lower()
            path = Path(dirpath) / name
            if ("detection" in low or low.startswith("voc")) and (path.is_dir() or low.endswith(ARCHIVE_EXT)):
                return path
    return None

def download_ip102_detection(dest):
    import gdown
    files = gdown.download_folder(IP102_FOLDER, skip_download=True, quiet=True, remaining_ok=True)
    print("Isi folder IP102:")
    for f in files:
        print("  ", f.path)
    det = [f for f in files if "detection" in f.path.lower() or "voc" in f.path.lower()]
    assert det, "File Detection tidak ditemukan di folder IP102. Kirim daftar isi folder di atas untuk disesuaikan."
    for f in det:
        out = Path(dest) / f.path
        out.parent.mkdir(parents=True, exist_ok=True)
        if not out.exists():
            gdown.download(id=f.id, output=str(out), quiet=False)
    return Path(dest) / Path(det[0].path).parts[0]

if DATA_SOURCE is None:
    DATA_SOURCE = find_detection_source(DRIVE if (USE_DRIVE or not IN_COLAB) else LOCAL / "ip102_download")
if DATA_SOURCE is None and not USE_DRIVE:
    DATA_SOURCE = download_ip102_detection(LOCAL / "ip102_download")
print("Sumber data:", DATA_SOURCE)
assert DATA_SOURCE is not None and Path(DATA_SOURCE).exists(), "Data Detection belum ditemukan."

In [ ]:
import shutil, tarfile, zipfile

EXTRACT = LOCAL / "ip102_detection"

def extract_all(src, dest):
    src = Path(src)
    dest.mkdir(parents=True, exist_ok=True)
    archives = [src] if src.is_file() else [p for p in src.rglob("*") if p.name.lower().endswith(ARCHIVE_EXT)]
    if not archives:                       # folder sudah berisi file terurai
        print("Menyalin folder", src, "→", dest)
        shutil.copytree(src, dest / src.name, dirs_exist_ok=True)
    for a in archives:
        print("Mengurai", a.name, "...")
        shutil.unpack_archive(str(a), str(dest))

if not any(EXTRACT.rglob("Annotations")):
    extract_all(DATA_SOURCE, EXTRACT)
print("Selesai. Isi:", [p.name for p in EXTRACT.iterdir()])

## Langkah 3 — Mengubah anotasi VOC → YOLO
IP102 memakai format **Pascal VOC** (file XML berisi koordinat kotak `xmin, ymin, xmax, ymax`). YOLO membutuhkan format teks: `kelas x_tengah y_tengah lebar tinggi`, semuanya dinormalisasi 0–1.

Hanya **14 kelas hama padi** (kelas 1–14 di `classes.txt` IP102) yang diambil. Pembagian data latih/uji mengikuti `ImageSets/Main/trainval.txt` dan `test.txt` dari IP102 bila tersedia, supaya hasilnya bisa dibandingkan dengan penelitian lain.

Sel berikut menulis file `voc2yolo.py` (ikut diunggah ke GitHub), lalu menjalankannya.

In [ ]:
%%writefile voc2yolo.py
import random, shutil, xml.etree.ElementTree as ET
from collections import Counter
from pathlib import Path

RICE_CLASSES = [
    "rice leaf roller", "rice leaf caterpillar", "paddy stem maggot", "asiatic rice borer",
    "yellow rice borer", "rice gall midge", "rice stemfly", "brown plant hopper",
    "white backed plant hopper", "small brown plant hopper", "rice water weevil",
    "rice leafhopper", "grain spreader thrips", "rice shell pest",
]


def find_voc_root(base):
    """Cari folder yang berisi Annotations/ dan JPEGImages/ di dalam base."""
    for ann in Path(base).rglob("Annotations"):
        if ann.is_dir() and (ann.parent / "JPEGImages").is_dir():
            return ann.parent
    raise FileNotFoundError(f"Folder Annotations/ + JPEGImages/ tidak ditemukan di {base}")


def class_index(name, zero_based):
    """IP102 menyimpan nama kelas sebagai angka (indeks kelas). Kembalikan indeks 0-based."""
    name = name.strip()
    if name.isdigit():
        i = int(name)
        return i if zero_based else i - 1
    low = name.lower()
    return RICE_CLASSES.index(low) if low in RICE_CLASSES else None


def detect_zero_based(ann_dir, sample=3000):
    names = set()
    for k, x in enumerate(Path(ann_dir).glob("*.xml")):
        if k >= sample:
            break
        for o in ET.parse(x).getroot().iter("object"):
            names.add(o.findtext("name", "").strip())
    nums = {int(n) for n in names if n.isdigit()}
    if not nums:
        return True
    return 0 in nums or max(nums) <= 101


def voc_to_yolo(voc_root, out_dir, val_ratio=0.2, seed=42):
    """Ubah anotasi VOC IP102 menjadi format YOLO, hanya untuk 14 kelas hama padi."""
    voc_root, out_dir = Path(voc_root), Path(out_dir)
    ann_dir, img_dir = voc_root / "Annotations", voc_root / "JPEGImages"
    zero_based = detect_zero_based(ann_dir)

    sets = voc_root / "ImageSets" / "Main"
    split_of = {}
    if (sets / "trainval.txt").exists() and (sets / "test.txt").exists():
        for s, f in (("train", "trainval.txt"), ("val", "test.txt")):
            for line in (sets / f).read_text().split():
                split_of[line.strip()] = s

    images = {p.stem: p for p in img_dir.iterdir() if p.suffix.lower() in (".jpg", ".jpeg", ".png")}
    kept, counts = [], Counter()
    for xml in sorted(ann_dir.glob("*.xml")):
        root = ET.parse(xml).getroot()
        w = float(root.findtext("size/width", "0") or 0)
        h = float(root.findtext("size/height", "0") or 0)
        img = images.get(xml.stem)
        if img is None:
            continue
        if w <= 0 or h <= 0:
            from PIL import Image
            with Image.open(img) as im:
                w, h = im.size
        lines = []
        for o in root.iter("object"):
            c = class_index(o.findtext("name", ""), zero_based)
            if c is None or not 0 <= c < len(RICE_CLASSES):
                continue
            b = o.find("bndbox")
            x1, y1, x2, y2 = (float(b.findtext(k)) for k in ("xmin", "ymin", "xmax", "ymax"))
            x1, x2 = max(0, min(x1, x2)), min(w, max(x1, x2))
            y1, y2 = max(0, min(y1, y2)), min(h, max(y1, y2))
            if x2 - x1 < 1 or y2 - y1 < 1:
                continue
            lines.append(f"{c} {(x1+x2)/2/w:.6f} {(y1+y2)/2/h:.6f} {(x2-x1)/w:.6f} {(y2-y1)/h:.6f}")
            counts[c] += 1
        if lines:
            kept.append((xml.stem, img, lines))

    rng = random.Random(seed)
    for stem, img, lines in kept:
        split = split_of.get(stem) or ("val" if rng.random() < val_ratio else "train")
        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)
        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)
        shutil.copy(img, out_dir / "images" / split / img.name)
        (out_dir / "labels" / split / f"{stem}.txt").write_text("\n".join(lines) + "\n")

    yaml = out_dir / "rice_pest.yaml"
    yaml.write_text(
        f"path: {out_dir.resolve()}\ntrain: images/train\nval: images/val\n"
        f"nc: {len(RICE_CLASSES)}\nnames:\n" + "".join(f"  {i}: {n}\n" for i, n in enumerate(RICE_CLASSES))
    )
    n_split = Counter(split_of.get(s) or "random" for s, _, _ in kept)
    return yaml, counts, len(kept), zero_based, n_split

In [ ]:
import importlib, voc2yolo
importlib.reload(voc2yolo)
from voc2yolo import RICE_CLASSES, find_voc_root, voc_to_yolo

VOC_ROOT = find_voc_root(EXTRACT)
YOLO_DIR = LOCAL / "rice_pest_yolo"
if not (YOLO_DIR / "rice_pest.yaml").exists():
    DATA_YAML, COUNTS, N_IMG, ZERO_BASED, N_SPLIT = voc_to_yolo(VOC_ROOT, YOLO_DIR)
    print("Indeks kelas 0-based:", ZERO_BASED, "| pembagian:", dict(N_SPLIT))
DATA_YAML = YOLO_DIR / "rice_pest.yaml"
for s in ("train", "val"):
    print(s, len(list((YOLO_DIR / "images" / s).glob("*"))), "gambar")

## Langkah 4 — Eksplorasi data
Dua hal penting untuk dipahami sebelum melatih:
1. **Ketidakseimbangan kelas**: kelas dengan contoh sedikit biasanya mendapat skor lebih rendah.
2. **Ukuran objek**: serangga sering hanya menempati bagian kecil gambar. Objek kecil adalah tantangan utama deteksi objek.

In [ ]:
import collections, numpy as np, matplotlib.pyplot as plt
ASSETS = WORK / "repo" / "assets"
ASSETS.mkdir(parents=True, exist_ok=True)

cls_count, areas = collections.Counter(), []
for f in (YOLO_DIR / "labels").rglob("*.txt"):
    for line in f.read_text().split("\n"):
        if line.strip():
            c, x, y, w, h = line.split()
            cls_count[int(c)] += 1
            areas.append(float(w) * float(h))

fig, ax = plt.subplots(1, 2, figsize=(15, 5))
idx = sorted(cls_count, key=cls_count.get)
ax[0].barh([RICE_CLASSES[i] for i in idx], [cls_count[i] for i in idx], color="#3B7A57")
ax[0].set_title("Jumlah objek per kelas / Objects per class")
ax[1].hist(np.sqrt(areas), bins=40, color="#C49A4B")
ax[1].set_title("Ukuran kotak relatif (√ luas) / Relative box size")
ax[1].set_xlabel("√(lebar × tinggi), 0–1")
plt.tight_layout(); plt.savefig(ASSETS / "data_overview.png", dpi=150); plt.show()
print("Total objek:", sum(cls_count.values()), "| objek kecil (<10% sisi gambar):",
      f"{np.mean(np.sqrt(areas) < 0.1):.1%}")

## Langkah 5 — Melatih YOLOv8n dan YOLOv8s
- **YOLOv8n** (*nano*, ±3 juta parameter): ringan dan cepat, cocok untuk perangkat kecil di lapangan.
- **YOLOv8s** (*small*, ±11 juta parameter): lebih akurat, tetapi lebih berat.

Keduanya memakai **transfer learning**: bobot awal dari COCO lalu dilatih ulang untuk hama padi. Dengan `USE_DRIVE = True`, hasil tersimpan di Google Drive dan bisa dilanjutkan dengan `RESUME = True` bila Colab terputus. Dengan `USE_DRIVE = False`, biarkan tab tetap terbuka; setelah pelatihan selesai, file `rice_pest_weights.zip` otomatis terunduh sebagai cadangan.

Perkiraan waktu di GPU T4: YOLOv8n ±30–60 menit, YOLOv8s ±1–1,5 jam untuk 50 epoch. Kalau waktu terbatas, turunkan `EPOCHS` menjadi 30.

In [ ]:
from ultralytics import YOLO

EPOCHS, IMGSZ, BATCH, RESUME = 50, 640, 16, False
RUNS = WORK / "runs"
MODELS = ["yolov8n", "yolov8s"]

def train(name):
    last = RUNS / name / "weights" / "last.pt"
    if RESUME and last.exists():
        return YOLO(str(last)).train(resume=True)
    if (RUNS / name / "weights" / "best.pt").exists():
        print(name, "sudah dilatih, dilewati."); return
    YOLO(f"{name}.pt").train(data=str(DATA_YAML), epochs=EPOCHS, imgsz=IMGSZ, batch=BATCH,
                             patience=15, seed=42, device=DEVICE, project=str(RUNS), name=name,
                             exist_ok=True, plots=True)

for m in MODELS:
    train(m)

if IN_COLAB and not USE_DRIVE:          # cadangan bobot model, supaya tidak hilang bila Colab terputus
    from google.colab import files
    shutil.make_archive(str(LOCAL / "rice_pest_weights"), "zip", RUNS)
    files.download(str(LOCAL / "rice_pest_weights.zip"))

## Langkah 6 — Evaluasi dan perbandingan
Metrik yang dipakai (sama dengan Tabel 3.6 di proposal):
- **Precision**: dari semua kotak yang ditebak model, berapa persen yang benar.
- **Recall**: dari semua hama yang sebenarnya ada, berapa persen yang ditemukan.
- **mAP@0,5**: rata-rata *average precision* semua kelas, dengan syarat kotak tebakan tumpang-tindih ≥50% (IoU ≥ 0,5) dengan kotak sebenarnya.
- **mAP@0,5:0,95**: sama, tetapi dirata-rata pada ambang IoU 0,5 sampai 0,95, jadi lebih ketat terhadap ketepatan posisi kotak.
- **Kecepatan**: waktu inferensi per gambar (ms).

In [ ]:
import pandas as pd
rows, per_class = [], {}
for m in MODELS:
    best = RUNS / m / "weights" / "best.pt"
    model = YOLO(str(best))
    r = model.val(data=str(DATA_YAML), split="val", imgsz=IMGSZ, batch=BATCH, device=DEVICE,
                  project=str(RUNS), name=f"{m}_val", exist_ok=True, plots=True, verbose=False)
    rows.append({"Model": m, "Parameters (M)": round(sum(p.numel() for p in model.model.parameters()) / 1e6, 1),
                 "Precision": round(r.box.mp, 3), "Recall": round(r.box.mr, 3),
                 "mAP@0.5": round(r.box.map50, 3), "mAP@0.5:0.95": round(r.box.map, 3),
                 "Inference (ms/img)": round(r.speed["inference"], 1),
                 "Size (MB)": round(best.stat().st_size / 1e6, 1)})
    idx = getattr(r.box, "ap_class_index", None)
    if idx is None or len(idx) == 0:
        idx = r.ap_class_index
    per_class[m] = {RICE_CLASSES[int(c)]: round(float(ap), 3) for c, ap in zip(idx, r.box.ap50)}

summary = pd.DataFrame(rows)
classes_df = pd.DataFrame(per_class).rename_axis("Class (AP@0.5)").sort_values(MODELS[-1], ascending=False)
summary.to_csv(WORK / "repo" / "results_summary.csv", index=False)
classes_df.to_csv(WORK / "repo" / "results_per_class.csv")
display(summary)
display(classes_df)

In [ ]:
from IPython.display import Image as IImage, display
best_name = summary.sort_values("mAP@0.5", ascending=False).iloc[0]["Model"]
print("Model terbaik (mAP@0.5):", best_name)
for pattern, out in (("confusion_matrix_normalized.png", "confusion_matrix_normalized.png"),
                     ("*PR_curve.png", "PR_curve.png")):
    found = sorted((RUNS / f"{best_name}_val").glob(pattern))
    if found:
        shutil.copy(found[0], ASSETS / out); display(IImage(str(found[0]), width=700))
p = RUNS / best_name / "results.png"
if p.exists():
    shutil.copy(p, ASSETS / "training_curves.png"); display(IImage(str(p), width=900))

## Langkah 7 — Contoh hasil deteksi
Delapan gambar uji diambil acak, lalu dideteksi dengan model terbaik.

In [ ]:
import random
random.seed(7)
val_imgs = sorted((YOLO_DIR / "images" / "val").glob("*"))
sample = random.sample(val_imgs, min(8, len(val_imgs)))
best_model = YOLO(str(RUNS / best_name / "weights" / "best.pt"))
res = best_model.predict([str(p) for p in sample], imgsz=IMGSZ, conf=0.25, device=DEVICE, verbose=False)
fig, axes = plt.subplots(2, 4, figsize=(18, 9))
for ax, r in zip(axes.ravel(), res):
    ax.imshow(r.plot()[..., ::-1]); ax.axis("off")
for ax in axes.ravel()[len(res):]:
    ax.axis("off")
plt.tight_layout(); plt.savefig(ASSETS / "sample_predictions.jpg", dpi=110); plt.show()

## Langkah 8 — Menyimpan hasil untuk GitHub
Sel ini menyiapkan folder `rice-pest-yolov8/repo` yang siap diunggah ke GitHub:
`README.md`, `voc2yolo.py`, `app.py`, `requirements.txt`, `assets/`, `weights/`, dan tabel hasil.
Notebook ini sendiri diunduh lewat **File › Download › Download .ipynb**, lalu ikut diunggah.

In [ ]:
REPO = WORK / "repo"
(REPO / "weights").mkdir(parents=True, exist_ok=True)
for m in MODELS:
    shutil.copy(RUNS / m / "weights" / "best.pt", REPO / "weights" / f"{m}_rice_pest.pt")
shutil.copy("voc2yolo.py", REPO / "voc2yolo.py")

def md_table(df):
    cols = list(df.columns)
    out = "| " + " | ".join(cols) + " |\n|" + "---|" * len(cols) + "\n"
    for _, r in df.iterrows():
        out += "| " + " | ".join(str(r[c]) for c in cols) + " |\n"
    return out

cls_table = md_table(classes_df.reset_index())
n_train = len(list((YOLO_DIR / "images" / "train").glob("*")))
n_val = len(list((YOLO_DIR / "images" / "val").glob("*")))

readme = f'''# Rice Pest Detection with YOLOv8

Object detection of **14 rice pest species** from the public **IP102** dataset using **YOLOv8**,
comparing a lightweight model (YOLOv8n) with a larger one (YOLOv8s) for field-deployable pest monitoring.

*Deteksi 14 jenis hama padi dari dataset publik IP102 menggunakan YOLOv8, dengan membandingkan model ringan
(YOLOv8n) dan model yang lebih besar (YOLOv8s) untuk pemantauan hama di lapangan.*

![Sample predictions](assets/sample_predictions.jpg)

## Results / Hasil
Validation on the IP102 rice subset ({n_train} training images, {n_val} test images, image size {IMGSZ}, {EPOCHS} epochs).

{md_table(summary)}
### Per-class AP@0.5
{cls_table}
| Confusion matrix | Precision-recall curve |
|---|---|
| ![](assets/confusion_matrix_normalized.png) | ![](assets/PR_curve.png) |

## Data / Dataset
- **IP102** (Wu et al., CVPR 2019), detection subset in Pascal VOC format, classes 1–14 (rice pests).
- Annotations converted to YOLO format with [`voc2yolo.py`](voc2yolo.py), keeping the official train/test split.
- The dataset is free for academic use and is **not redistributed** in this repository.

![Data overview](assets/data_overview.png)

## Method / Metode
1. Convert Pascal VOC annotations to YOLO format (14 rice pest classes only).
2. Fine-tune COCO-pretrained YOLOv8n and YOLOv8s (transfer learning).
3. Evaluate with precision, recall, mAP@0.5, mAP@0.5:0.95, and inference time.

## Run it / Menjalankan
- **Notebook:** open `rice_pest_yolov8.ipynb` in Google Colab (T4 GPU) and run the cells in order.
- **Demo:** `pip install -r requirements.txt` then `python app.py`.

## Files
| File | Description |
|---|---|
| `rice_pest_yolov8.ipynb` | Full pipeline: data preparation, training, evaluation, demo |
| `voc2yolo.py` | IP102 VOC → YOLO converter for the rice subset |
| `app.py` | Gradio web demo |
| `weights/` | Trained weights (best checkpoints) |
| `results_summary.csv`, `results_per_class.csv` | Evaluation results |

## Acknowledgements
- Wu, X., Zhan, C., Lai, Y.-K., Cheng, M.-M., & Yang, J. (2019). IP102: A large-scale benchmark dataset for insect pest recognition. *CVPR*.
- [Ultralytics YOLOv8](https://github.com/ultralytics/ultralytics) (AGPL-3.0).

## License
Code is released under the **AGPL-3.0** license, following Ultralytics YOLOv8.

---
**Desi Susanti, S.Kom., M.Kom.** · [Portfolio](https://des1susanti.github.io/portofolio-v2/index.html) · [LinkedIn](https://www.linkedin.com/in/desi-susanti-70a7b831a/)
'''
(REPO / "README.md").write_text(readme)
print("README.md dibuat. Isi folder repo:")
for p in sorted(REPO.rglob("*")):
    if p.is_file():
        print(" ", p.relative_to(REPO), f"({p.stat().st_size/1e6:.1f} MB)")

## Langkah 9 — Demo Gradio
Sel pertama menulis `app.py` (untuk GitHub dan Hugging Face Spaces). Sel kedua menjalankan demo langsung di Colab; klik tautan **public URL** yang muncul untuk membukanya di tab baru atau di HP.

In [ ]:
app_code = """import gradio as gr
from ultralytics import YOLO

model = YOLO("weights/yolov8n_rice_pest.pt")


def detect(image, conf):
    if image is None:
        return None, []
    r = model.predict(image, conf=conf, imgsz=640, verbose=False)[0]
    rows = [[r.names[int(c)], round(float(s), 3)] for c, s in zip(r.boxes.cls, r.boxes.conf)]
    return r.plot()[..., ::-1], rows


demo = gr.Interface(
    fn=detect,
    inputs=[gr.Image(type="pil", label="Upload a rice pest photo / Unggah foto hama padi"),
            gr.Slider(0.05, 0.95, value=0.25, step=0.05, label="Confidence threshold")],
    outputs=[gr.Image(label="Detections / Hasil deteksi"),
             gr.Dataframe(headers=["Pest / Hama", "Confidence"], label="Detected objects")],
    title="Rice Pest Detection with YOLOv8",
    description="Detects 14 rice pest species (IP102 dataset). By Desi Susanti, S.Kom., M.Kom.",
)

if __name__ == "__main__":
    demo.launch()
"""
(REPO / "app.py").write_text(app_code)
(REPO / "requirements.txt").write_text("ultralytics>=8.3,<8.4\ngradio>=4.44\n")
print("app.py dan requirements.txt dibuat di", REPO)

In [ ]:
import gradio as gr
demo_model = YOLO(str(REPO / "weights" / "yolov8n_rice_pest.pt"))

def detect(image, conf):
    if image is None:
        return None, []
    r = demo_model.predict(image, conf=conf, imgsz=IMGSZ, device=DEVICE, verbose=False)[0]
    rows = [[r.names[int(c)], round(float(s), 3)] for c, s in zip(r.boxes.cls, r.boxes.conf)]
    return r.plot()[..., ::-1], rows

gr.Interface(
    fn=detect,
    inputs=[gr.Image(type="pil", label="Unggah foto hama padi"),
            gr.Slider(0.05, 0.95, value=0.25, step=0.05, label="Confidence threshold")],
    outputs=[gr.Image(label="Hasil deteksi"), gr.Dataframe(headers=["Hama", "Confidence"])],
    title="Rice Pest Detection with YOLOv8",
    examples=[[str(p), 0.25] for p in sample[:4]],
).launch(share=True, debug=False)

In [ ]:
# Unduh seluruh folder repo (README, gambar hasil, bobot, app.py) sebagai zip untuk diunggah ke GitHub
shutil.make_archive(str(LOCAL / "rice-pest-yolov8-repo"), "zip", REPO)
if IN_COLAB:
    from google.colab import files
    files.download(str(LOCAL / "rice-pest-yolov8-repo.zip"))
print("Zip:", LOCAL / "rice-pest-yolov8-repo.zip")

## Selesai — langkah berikutnya
1. **Unduh notebook ini**: File › Download › Download .ipynb, beri nama `rice_pest_yolov8.ipynb`.
2. **Buat repositori GitHub** `rice-pest-yolov8`, lalu unggah notebook dan seluruh isi file `rice-pest-yolov8-repo.zip` (setelah diekstrak).
3. **Lisensi**: Add file › Create new file › `LICENSE` › Choose a license template › **GNU AGPLv3**.
4. **Demo permanen (opsional)**: buat Space baru di huggingface.co (SDK: Gradio), unggah `app.py`, `requirements.txt`, dan folder `weights/`.

### Pertanyaan yang mungkin muncul saat wawancara
- *Mengapa YOLOv8?* Detektor satu tahap yang cepat dan akurat, mendukung *transfer learning*, dan ukurannya bisa dipilih sesuai perangkat.
- *Apa beda mAP@0,5 dan mAP@0,5:0,95?* Yang kedua lebih ketat karena menuntut posisi kotak lebih tepat.
- *Mengapa kelas tertentu skornya rendah?* Lihat grafik jumlah objek per kelas dan *confusion matrix*: biasanya karena contohnya sedikit, objeknya kecil, atau mirip dengan kelas lain (misalnya jenis-jenis wereng).
- *Pilih YOLOv8n atau YOLOv8s untuk lapangan?* Bandingkan selisih mAP dengan selisih kecepatan dan ukuran model di tabel hasil.